In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run ../1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar-dp-umesh/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

#define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df = (spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("Total_Rows: ", df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
df.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true")\
 .mode("append")\
 .saveAsTable(bronze_table)